In [1]:
from __future__ import annotations
from pathlib import Path
import os, sys, json, shutil, logging
from datetime import datetime
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
try:
    import yaml
except Exception as e:
    raise RuntimeError("pip install pyyaml") from e
 
 
def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for c in [here, *here.parents]:
        if c.name == "fruitnet-chili-yield":
            return c
    return here
 
 
PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
 
# ---- ĐƯỜNG DẪN (WSL) ----
DATA_DIR = Path(os.environ.get("DATA_DIR", "/mnt/f/fruitnet-chili-yield-data/finetune_data")).resolve()
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", "/mnt/f/fruitnet-chili-yield-outputs")).resolve()
COCO_PRETRAIN_DIR = Path(os.environ.get("COCO_PRETRAIN_DIR",
                                        str(OUTPUT_ROOT / "runs/detectors/coco_pretrain"))).resolve()
DATASET_NAME = DATA_DIR.name  # "finetune_data"
 
EXP_DIR = OUTPUT_ROOT / "results" / f"exp02_{DATASET_NAME}"
RUN_DIR = OUTPUT_ROOT / "runs" / "detectors" / DATASET_NAME
VAL_DIR = OUTPUT_ROOT / "runs" / "detectors" / f"{DATASET_NAME}_val"
ARTIFACT_DIR = OUTPUT_ROOT / "artifacts" / "finetuned"
for d in ["logs", EXP_DIR, RUN_DIR, VAL_DIR, ARTIFACT_DIR, Path("configs/data")]:
    Path(d).mkdir(parents=True, exist_ok=True)
 
LOG_PATH = Path("logs") / f"06_train_{datetime.now():%Y%m%d_%H%M%S}.log"
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s",
                    handlers=[logging.FileHandler(LOG_PATH, encoding="utf-8"), logging.StreamHandler(sys.stdout)])
 
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
CLASS_MODE = os.environ.get("CLASS_MODE", "single_chili").strip()
SEED = int(os.environ.get("SEED", "42"))
 
 
# ---- Tạo/định vị data yaml trỏ tới DATA_DIR ----
def default_names():
    return ["chili"] if CLASS_MODE == "single_chili" else ["Chili_Green", "Chili_Transition", "Chili_Red"]
 
 
def resolve_data_yaml() -> Path:
    names = default_names()
    src_yaml = DATA_DIR / "data.yaml"
    if src_yaml.exists():
        try:
            d = yaml.safe_load(src_yaml.read_text(encoding="utf-8")) or {}
            nm = d.get("names")
            if isinstance(nm, dict):
                names = [nm[k] for k in sorted(nm, key=lambda x: int(x))]
            elif isinstance(nm, list) and nm:
                names = nm
        except Exception as e:
            logging.warning("Đọc %s lỗi, dùng names mặc định: %s", src_yaml, e)
    out = Path("configs/data") / f"{DATASET_NAME}.yaml"
    out.write_text(yaml.safe_dump(
        {"path": str(DATA_DIR), "train": "images/train", "val": "images/val", "test": "images/test",
         "nc": len(names), "names": {i: n for i, n in enumerate(names)}},
        sort_keys=False, allow_unicode=True), encoding="utf-8")
    if not (DATA_DIR / "images" / "train").exists():
        logging.warning("Không thấy %s — kiểm tra lại DATA_DIR/layout.", DATA_DIR / "images" / "train")
    return out
 
 
DATA_YAML = resolve_data_yaml()
 
# ---- Augmentation online từ 05 (tên có thể là finetune_data_aug hoặc finetune_data_1_aug) ----
AUG_KEYS = {"hsv_h", "hsv_s", "hsv_v", "degrees", "translate", "scale", "shear", "perspective",
            "flipud", "fliplr", "mosaic", "close_mosaic", "mixup", "copy_paste", "erasing", "auto_augment"}
 
 
def load_aug_hyp() -> dict:
    for name in [f"{DATASET_NAME}_aug.yaml", "finetune_data_1_aug.yaml", "finetune_data_aug.yaml"]:
        p = Path("configs/aug") / name
        if p.exists():
            raw = yaml.safe_load(p.read_text(encoding="utf-8")) or {}
            logging.info("Nạp augmentation online từ %s", p)
            return {k: v for k, v in raw.items() if k in AUG_KEYS}
    logging.warning("Không thấy configs/aug/*_aug.yaml (chạy 05). Dùng aug mặc định Ultralytics.")
    return {}
 
 
IMG_SIZE = int(os.environ.get("IMG_SIZE", "640"))
BATCH = int(os.environ.get("BATCH", "8"))
EPOCHS = int(os.environ.get("EPOCHS", "100"))
WORKERS = int(os.environ.get("WORKERS", "8"))
DEVICE = os.environ.get("DEVICE", "0")
PATIENCE = int(os.environ.get("PATIENCE", "30"))
RUN_TRAINING = os.environ.get("RUN_TRAINING", "1") == "1"
RUN_VALIDATION = os.environ.get("RUN_VALIDATION", "1") == "1"
RUN_COUNT_EVAL = os.environ.get("RUN_COUNT_EVAL", "1") == "1"
EVAL_SPLIT_REQUESTED = os.environ.get("EVAL_SPLIT", "test").strip()
 
# Model list khớp folder trong coco_pretrain: yolov8n, yolov5s, fruitnet_b/g/gc/gcs
MODEL_CANDIDATES = [
    {"model_name": "YOLOv8n", "model_key": "yolov8n", "fallback": "yolov8n.pt"},
    {"model_name": "YOLOv5s", "model_key": "yolov5s", "fallback": "yolov5su.pt"},
    {"model_name": "FruitNet B", "model_key": "fruitnet_b", "fallback": "configs/models/fruitnet_b.yaml"},
    {"model_name": "FruitNet G", "model_key": "fruitnet_g", "fallback": "configs/models/fruitnet_g.yaml"},
    {"model_name": "FruitNet GC", "model_key": "fruitnet_gc", "fallback": "configs/models/fruitnet_gc.yaml"},
    {"model_name": "FruitNet GCS", "model_key": "fruitnet_gcs", "fallback": "configs/models/fruitnet_gcs.yaml"},
]
for m in MODEL_CANDIDATES:
    m["source"] = str(COCO_PRETRAIN_DIR / m["model_key"] / "weights" / "best.pt")
mk = os.environ.get("MODEL_KEYS", "").strip()
if mk:
    allow = {x.strip() for x in mk.split(",") if x.strip()}
    MODEL_CANDIDATES = [m for m in MODEL_CANDIDATES if m["model_key"] in allow]
 
 
def register_custom_modules() -> bool:
    try:
        from models.fruitnet.register_ultralytics_modules import register_fruitnet_modules
        register_fruitnet_modules()
        logging.info("Registered FruitNet custom modules (CoordAtt).")
        return True
    except Exception as e:
        logging.warning("Không import được FruitNet modules (baseline YOLO vẫn chạy): %s", repr(e))
        return False
 
 
def iter_images(path: Path):
    if path.exists():
        for p in path.rglob("*"):
            if p.is_file() and p.suffix.lower() in IMG_EXTS:
                yield p
 
 
def split_image_count(split: str) -> int:
    cfg = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
    return len(list(iter_images(Path(cfg["path"]) / cfg[split])))
 
 
def has_train_and_val() -> bool:
    return split_image_count("train") > 0 and split_image_count("val") > 0
 
 
def resolve_eval_split(preferred="test"):
    for s in [preferred, "test", "val", "train"]:
        if s in {"train", "val", "test"} and split_image_count(s) > 0:
            return s
    return None
 
 
def choose_model_source(spec):
    src = Path(spec["source"])
    if src.exists():
        return str(src)
    logging.warning("Không thấy COCO weight %s -> dùng fallback %s", src, spec["fallback"])
    fb = spec["fallback"]
    if fb.endswith(".pt") and not fb.startswith("configs/"):
        return fb
    if Path(fb).exists():
        return fb
    return None
 
 
def yaml_needs_coordatt(uri):
    p = Path(uri) if uri else None
    if not p or not p.exists() or p.suffix.lower() not in {".yaml", ".yml"}:
        return False
    t = p.read_text(encoding="utf-8", errors="ignore")
    return "CoordAtt" in t or "CoordinateAttention" in t
 
 
def find_best_last(run_dir):
    b, l = run_dir / "weights" / "best.pt", run_dir / "weights" / "last.pt"
    return (b if b.exists() else None, l if l.exists() else None)
 
 
def metrics_to_dict(m):
    box = getattr(m, "box", None)
    if box is None:
        return {"precision": np.nan, "recall": np.nan, "map50": np.nan, "map50_95": np.nan}
    g = lambda a: float(getattr(box, a, np.nan))
    return {"precision": g("mp"), "recall": g("mr"), "map50": g("map50"), "map50_95": g("map")}
 
 
def model_complexity(model):
    try:
        info = model.info(verbose=False)
        return float(info[1]) / 1e6, (float(info[3]) if len(info) > 3 else float("nan"))
    except Exception:
        return float("nan"), float("nan")
 
 
def train_one(spec, custom_ok, aug_hyp):
    from ultralytics import YOLO
    uri = choose_model_source(spec)
    row = {"model_name": spec["model_name"], "model_key": spec["model_key"], "model_uri_used": uri,
           "data_yaml": str(DATA_YAML), "status": "unknown", "best_pt": "", "exported_best": "",
           "eval_split": "", "precision": np.nan, "recall": np.nan, "map50": np.nan,
           "map50_95": np.nan, "params_M": np.nan, "GFLOPs": np.nan, "error": ""}
    if uri is None:
        row["status"] = "skipped_missing_model"; return row
    if yaml_needs_coordatt(uri) and not custom_ok:
        row["status"] = "skipped_coordatt_not_registered"
        row["error"] = "fruitnet_*.yaml dùng CoordAtt nhưng register_fruitnet_modules thất bại."
        return row
    if RUN_TRAINING and not has_train_and_val():
        row["status"] = "skipped_dataset_not_ready"; row["error"] = "train/val rỗng."; return row
    eval_split = None
    if RUN_VALIDATION:
        eval_split = resolve_eval_split(EVAL_SPLIT_REQUESTED)
        if eval_split is None:
            row["status"] = "skipped_no_eval_split"; return row
        row["eval_split"] = eval_split
 
    run_name = f"{spec['model_key']}_{DATASET_NAME}"
    try:
        model = YOLO(uri)
        row["params_M"], row["GFLOPs"] = model_complexity(model)
        if RUN_TRAINING:
            model.train(data=str(DATA_YAML), imgsz=IMG_SIZE, batch=BATCH, epochs=EPOCHS,
                        workers=WORKERS, device=DEVICE, seed=SEED, patience=PATIENCE,
                        project=str(RUN_DIR), name=run_name, exist_ok=True, pretrained=True,
                        plots=True, save=True, **aug_hyp)
        best, _ = find_best_last(RUN_DIR / run_name)
        row["best_pt"] = str(best or "")
        if best:
            exp = ARTIFACT_DIR / f"{spec['model_key']}_{DATASET_NAME}_best.pt"
            shutil.copy2(best, exp); row["exported_best"] = str(exp); val_uri = str(exp)
        else:
            val_uri = uri
        if RUN_VALIDATION:
            metrics = YOLO(val_uri).val(data=str(DATA_YAML), imgsz=IMG_SIZE, batch=BATCH,
                                        workers=WORKERS, device=DEVICE, split=eval_split,
                                        project=str(VAL_DIR), name=f"{run_name}_{eval_split}",
                                        exist_ok=True, plots=True)
            row.update(metrics_to_dict(metrics))
        row["status"] = "ok"
    except Exception as e:
        row["status"] = "failed"; row["error"] = repr(e)
        logging.exception("Train/val failed: %s", spec["model_name"])
    return row
 
 
def count_gt(lbl, target_cls):
    if not lbl.exists():
        return 0
    n = 0
    for ln in lbl.read_text(encoding="utf-8", errors="ignore").splitlines():
        p = ln.split()
        if len(p) == 5:
            try:
                if int(float(p[0])) == target_cls:
                    n += 1
            except Exception:
                pass
    return n
 
 
def evaluate_count(model_path, split="test", conf=0.25, iou=0.70, target_cls=None, max_images=None):
    from ultralytics import YOLO
    cfg = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8")); root = Path(cfg["path"])
    if split_image_count(split) == 0:
        split = resolve_eval_split(split)
        if split is None:
            raise RuntimeError("Không có split để count-eval.")
    if target_cls is None:
        target_cls = 0 if CLASS_MODE == "single_chili" else 2
    imgs = list(iter_images(root / cfg[split]))
    if max_images:
        imgs = imgs[:max_images]
    model = YOLO(model_path); rows = []
    for img in tqdm(imgs, desc=f"count-eval:{split}"):
        r = model.predict(str(img), imgsz=IMG_SIZE, conf=conf, iou=iou, verbose=False)[0]
        pred = 0
        if r.boxes is not None and r.boxes.cls is not None:
            pred = sum(1 for c in r.boxes.cls.cpu().numpy().tolist() if int(c) == target_cls)
        gt = count_gt(root / "labels" / split / f"{img.stem}.txt", target_cls)
        rows.append({"image": img.name, "gt": gt, "pred": pred, "abs_err": abs(pred - gt),
                     "sq_err": (pred - gt) ** 2, "ape": abs(pred - gt) / max(gt, 1)})
    return pd.DataFrame(rows), split
 
 
def summarize_count(df):
    if df.empty:
        return {}
    mape = float(df["ape"].mean() * 100)
    return {"MAE": float(df["abs_err"].mean()), "RMSE": float(np.sqrt(df["sq_err"].mean())),
            "MAPE": mape, "Counting_Accuracy": max(0.0, 100 - mape)}
 
 
def main():
    print("DATA_DIR:", DATA_DIR, "| DATA_YAML:", DATA_YAML)
    print("COCO_PRETRAIN_DIR:", COCO_PRETRAIN_DIR)
    print("OUTPUT_ROOT:", OUTPUT_ROOT)
    print("MODELS:", [m["model_key"] for m in MODEL_CANDIDATES])
    custom_ok = register_custom_modules()
    aug_hyp = load_aug_hyp()
    print("Augmentation online:", aug_hyp or "(mặc định)")
 
    rows = []
    for spec in MODEL_CANDIDATES:
        rows.append(train_one(spec, custom_ok, aug_hyp))
        pd.DataFrame(rows).to_csv(EXP_DIR / f"{DATASET_NAME}_finetune_results.csv", index=False)
    df = pd.DataFrame(rows)
    print("\n=== Detection results ===")
    print(df[["model_name", "status", "eval_split", "precision", "recall",
              "map50", "map50_95", "params_M", "GFLOPs"]].to_string(index=False))
 
    if RUN_COUNT_EVAL:
        crows = []
        for r in rows:
            mp = r.get("exported_best") or r.get("best_pt")
            if r["status"] == "ok" and mp and Path(mp).exists():
                try:
                    cdf, used = evaluate_count(mp, split=EVAL_SPLIT_REQUESTED)
                    s = summarize_count(cdf); s.update({"model": r["model_name"], "split": used})
                    cdf.to_csv(EXP_DIR / f"{r['model_key']}_count_eval.csv", index=False)
                    crows.append(s)
                except Exception as e:
                    logging.warning("Count-eval lỗi %s: %s", r["model_name"], repr(e))
        if crows:
            pd.DataFrame(crows).to_csv(EXP_DIR / f"{DATASET_NAME}_count_summary.csv", index=False)
            print("\n=== Counting baseline (fixed threshold, trước RL) ===")
            print(pd.DataFrame(crows).to_string(index=False))
    print("\n[06] Done ->", EXP_DIR, "| Tiếp theo: 07_rl_count_refine.py")
 
 
if __name__ == "__main__":
    main()

/home/diy-hus/miniconda3/envs/fruitnet/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DATA_DIR: /mnt/f/fruitnet-chili-yield-data/finetune_data | DATA_YAML: configs/data/finetune_data.yaml
COCO_PRETRAIN_DIR: /mnt/f/fruitnet-chili-yield-outputs/runs/detectors/coco_pretrain
OUTPUT_ROOT: /mnt/f/fruitnet-chili-yield-outputs
MODELS: ['yolov8n', 'yolov5s', 'fruitnet_b', 'fruitnet_g', 'fruitnet_gc', 'fruitnet_gcs']
2026-06-27 10:46:27,434 | INFO | Registered FruitNet custom modules (CoordAtt).
2026-06-27 10:46:27,435 | WARNING | Không thấy configs/aug/*_aug.yaml (chạy 05). Dùng aug mặc định Ultralytics.
Augmentation online: (mặc định)
New https://pypi.org/project/ultralytics/8.4.80 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.60 🚀 Python-3.10.20 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050, 8192MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_

/home/diy-hus/miniconda3/envs/fruitnet/lib/python3.10/site-packages/torch/autograd/graph.py:869: UserWarning: adaptive_avg_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/100      2.57G      1.431      1.463      1.649          4        640: 100% ━━━━━━━━━━━━ 1366/1366 2.8it/s 8:02<0.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 5.2it/s 6.9s0.2s
                   all        569       1712      0.811      0.768      0.841      0.527

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/100      2.57G      1.231      1.042       1.48          8        640: 0% ──────────── 0/1366  0.2s

/home/diy-hus/miniconda3/envs/fruitnet/lib/python3.10/site-packages/torch/autograd/graph.py:869: UserWarning: adaptive_avg_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


...

     98/100      2.58G     0.4402     0.2341     0.9091          4        640: 100% ━━━━━━━━━━━━ 1366/1366 3.9it/s 5:51<0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 5.2it/s 6.9s0.2s
                   all        569       1712      0.948      0.907      0.937      0.755

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     99/100      2.58G     0.3656      0.181     0.8765          8        640: 0% ──────────── 0/1366  0.2s

/home/diy-hus/miniconda3/envs/fruitnet/lib/python3.10/site-packages/torch/autograd/graph.py:869: UserWarning: adaptive_avg_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     99/100      2.58G      0.439     0.2341     0.9097          4        640: 100% ━━━━━━━━━━━━ 1366/1366 3.9it/s 5:46<0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 5.0it/s 7.3s0.2s
                   all        569       1712      0.948      0.907      0.938      0.756
EarlyStopping: Training stopped early as no improvement observed in last 30 epochs. Best results observed at epoch 69, best model saved as best.pt.
To update EarlyStopping(patience=30) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

99 epochs completed in 9.873 hours.
Optimizer stripped from /mnt/f/fruitnet-chili-yield-outputs/runs/detectors/finetune_data/fruitnet_gcs_finetune_data/weights/last.pt, 28.1MB
Optimizer stripped from /mnt/f/fruitnet-chili-yield-outputs/runs/detectors/finetune_data/fruitnet_gcs_finetune_data/weights/best.pt, 28.1MB

Validating /mnt/f/fruitnet-chili-yield-outputs/runs/detect

count-eval:test: 100%|██████████| 115/115 [00:04<00:00, 28.44it/s]


=== Counting baseline (fixed threshold, trước RL) ===
     MAE     RMSE      MAPE  Counting_Accuracy        model split
0.478261 1.379981 12.909248          87.090752      YOLOv8n  test
0.530435 1.335144 17.222912          82.777088      YOLOv5s  test
0.504348 1.571485 14.946515          85.053485   FruitNet B  test
0.539130 1.577009 12.528672          87.471328   FruitNet G  test
0.617391 1.752017 15.686680          84.313320  FruitNet GC  test
0.469565 1.414214 13.596305          86.403695 FruitNet GCS  test

[06] Done -> /mnt/f/fruitnet-chili-yield-outputs/results/exp02_finetune_data | Tiếp theo: 07_rl_count_refine.py
